# Notebook 1 — Collecte et Préparation des Données (ETL)

## Périmètre de recherche

Ce notebook implémente le pipeline ETL (*Extract, Transform, Load*) du Chapitre 4.  
Il collecte les données brutes depuis deux sources officielles, les transforme, puis  
produit le dataset consolidé utilisé par les notebooks de modélisation.

**Sources :**
- **NVD (National Vulnerability Database)** — API v2.0 (NIST) → métriques CVSS v3.1
- **CISA KEV (Known Exploited Vulnerabilities)** → vérité terrain (`is_exploited`)

**CWE ciblées — 7 faiblesses Auth/Authz :**

| CWE | Intitulé |
|-----|----------|
| CWE-287 | Improper Authentication |
| CWE-862 | Missing Authorization |
| CWE-863 | Incorrect Authorization |
| CWE-306 | Missing Authentication for Critical Function |
| CWE-307 | Improper Restriction of Excessive Authentication Attempts |
| CWE-798 | Use of Hard-coded Credentials |
| CWE-613 | Insufficient Session Expiration |

**Output :** `data/api_vulnerabilities_processed.csv` (N × 14 colonnes)

---
> **Note d'exécution :** lancez les cellules dans l'ordre numérique.  
> Si `data/nvd_api_raw_vulnerabilities.json` existe déjà, la collecte NVD  
> est ignorée (lecture du checkpoint disque). Supprimez ce fichier pour forcer  
> une collecte fraîche depuis l'API NVD.

In [ ]:
# ==============================================================================
# CELLULE 1 — Imports, configuration et résolution des chemins
# ==============================================================================

import os
import time
import json
import logging
import warnings
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Any

warnings.filterwarnings('ignore')

# ── Configuration du logging ─────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler()]
)

# ── Résolution du répertoire de données ──────────────────────────────────────
# Compatible avec une exécution depuis src/ (VSCode) ou depuis la racine.
_cwd = Path(os.getcwd())
DATA_DIR = (_cwd.parent / 'data') if _cwd.name == 'src' else (_cwd / 'data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_FILE  = DATA_DIR / 'nvd_api_raw_vulnerabilities.json'
OUTPUT_CSV       = DATA_DIR / 'api_vulnerabilities_processed.csv'

# ── URLs et constantes API ────────────────────────────────────────────────────
CISA_KEV_URL      = 'https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json'
NVD_API_BASE_URL  = 'https://services.nvd.nist.gov/rest/json/cves/2.0'
MAX_RESULTS_PAGE  = 2000  # Maximum autorisé par l'API NVD v2.0

# ── Périmètre Auth/Authz — 7 CWE cibles ─────────────────────────────────────
TARGET_API_CWES = [
    'CWE-287',  # Improper Authentication
    'CWE-862',  # Missing Authorization
    'CWE-863',  # Incorrect Authorization
    'CWE-306',  # Missing Authentication for Critical Function
    'CWE-307',  # Improper Restriction of Excessive Authentication Attempts
    'CWE-798',  # Use of Hard-coded Credentials
    'CWE-613',  # Insufficient Session Expiration
]

# ── Features CVSS à encoder (identiques dans les notebooks 2 et 3) ───────────
CVSS_FEATURES = [
    'attack_vector', 'attack_complexity',
    'privileges_required', 'user_interaction', 'scope'
]

# ── Date de référence pour le calcul de l'âge des CVE ───────────────────────
# Utiliser pd.Timestamp.today() pour un calcul dynamique.
# Pour reproduire exactement les résultats du mémoire : pd.Timestamp('2026-06-15')
REFERENCE_DATE = pd.Timestamp.today().normalize()

logging.info(f'Répertoire de données : {DATA_DIR}')
logging.info(f'Date de référence     : {REFERENCE_DATE.date()}')
logging.info(f'CWE ciblées           : {TARGET_API_CWES}')

## Section 1 — Collecte de la Vérité Terrain (CISA KEV)

Le catalogue CISA KEV (*Known Exploited Vulnerabilities*) recense les CVE  
pour lesquelles une exploitation active dans la nature a été documentée.  
Ces identifiants constituent la variable cible `is_exploited` (Y=1).

In [ ]:
# ==============================================================================
# CELLULE 2 — Téléchargement du catalogue CISA KEV
# ==============================================================================

def fetch_cisa_kev(url: str) -> set:
    """
    Télécharge le catalogue CISA KEV et retourne l'ensemble des CVE-IDs activement
    exploités dans la nature.

    Parameters
    ----------
    url : str
        URL du flux JSON officiel CISA.

    Returns
    -------
    set[str]
        Ensemble des identifiants CVE (ex. 'CVE-2021-44228').
    """
    logging.info('Téléchargement du catalogue CISA KEV...')
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    data = response.json()
    cve_set = {v['cveID'] for v in data.get('vulnerabilities', []) if 'cveID' in v}
    logging.info(f'[✓] CISA KEV chargé : {len(cve_set):,} CVE activement exploités')
    return cve_set


cisa_exploited_cves = fetch_cisa_kev(CISA_KEV_URL)

## Section 2 — Collecte depuis l'API NVD

L'API NVD v2.0 impose un *rate-limit* de 5 requêtes / 30 s sans clé API  
(50 req/30 s avec clé). La fonction ci-dessous pagine automatiquement  
et respecte cette contrainte.

> **Checkpoint :** si `data/nvd_api_raw_vulnerabilities.json` existe déjà,  
> la collecte est ignorée. Supprimez ce fichier pour forcer une re-collecte.

In [ ]:
# ==============================================================================
# CELLULE 3 — Fonction de collecte NVD (paginée, résiliente)
# ==============================================================================

def fetch_nvd_api(base_url: str,
                  target_cwes: List[str],
                  checkpoint_path: Path,
                  max_per_page: int = 2000) -> List[Dict[str, Any]]:
    """
    Interroge l'API NVD v2.0 de manière paginée pour chaque CWE cible.

    - Réutilise le checkpoint JSON si disponible (évite les appels API répétés).
    - Gère le rate-limit HTTP 403/429 avec une temporisation automatique.
    - Sauvegarde le checkpoint après chaque CWE traitée.

    Parameters
    ----------
    base_url : str
        Endpoint de l'API NVD.
    target_cwes : list[str]
        Liste des identifiants CWE à collecter (ex. ['CWE-287', 'CWE-862']).
    checkpoint_path : Path
        Chemin vers le fichier JSON de cache.
    max_per_page : int
        Nombre maximum de résultats par page (2000 = maximum NVD).

    Returns
    -------
    list[dict]
        Liste brute des objets CVE retournés par l'API.
    """
    # ── Lecture du checkpoint si disponible ──────────────────────────────────
    if checkpoint_path.exists():
        with open(checkpoint_path, 'r', encoding='utf-8') as f:
            cached = json.load(f)
        logging.info(f'[✓] Checkpoint NVD détecté : {len(cached):,} CVE rechargées '
                     f'(supprimez {checkpoint_path.name} pour une re-collecte)')
        return cached

    all_items = []

    for cwe in target_cwes:
        start_idx   = 0
        total_items = 1  # Mis à jour après le premier appel

        logging.info(f'Collecte NVD pour {cwe}...')

        while start_idx < total_items:
            params = {
                'cweId'          : cwe,
                'resultsPerPage' : max_per_page,
                'startIndex'     : start_idx,
            }

            try:
                resp = requests.get(base_url, params=params, timeout=30)

                # Rate-limit — attente de 30 s avant nouvelle tentative
                if resp.status_code in (403, 429):
                    logging.warning('Rate-limit NVD détecté — pause 30 s...')
                    time.sleep(30)
                    continue

                resp.raise_for_status()
                payload     = resp.json()
                total_items = payload.get('totalResults', 0)
                page_items  = payload.get('vulnerabilities', [])

                all_items.extend(v.get('cve', {}) for v in page_items)
                start_idx  += len(page_items)

                logging.info(f'  [{cwe}] {start_idx}/{total_items} CVE collectées')

                # Pause obligatoire : respecte le rate-limit NVD sans clé API
                time.sleep(6.5)

            except requests.exceptions.RequestException as e:
                logging.error(f'Erreur réseau pour {cwe} (offset={start_idx}) : {e}')
                time.sleep(15)

        # Sauvegarde intermédiaire après chaque CWE (résilience en cas d'interruption)
        with open(checkpoint_path, 'w', encoding='utf-8') as f:
            json.dump(all_items, f)
        logging.info(f'  [{cwe}] Checkpoint sauvegardé ({len(all_items):,} CVE au total)')

    logging.info(f'[✓] Collecte NVD terminée : {len(all_items):,} enregistrements bruts')
    return all_items

In [ ]:
# ==============================================================================
# CELLULE 4 — Exécution de la collecte NVD
# ==============================================================================

# Durée estimée : 5–25 min sans clé API (rate-limit 5 req/30 s)
#                 1–3 min avec clé API NVD (50 req/30 s)
raw_nvd_data = fetch_nvd_api(
    base_url       = NVD_API_BASE_URL,
    target_cwes    = TARGET_API_CWES,
    checkpoint_path= CHECKPOINT_FILE,
    max_per_page   = MAX_RESULTS_PAGE
)

print(f'[+] Données NVD brutes : {len(raw_nvd_data):,} enregistrements')

## Section 3 — Parsing, Transformation et Fusion

Extraction des métriques CVSS v3.1 depuis les structures JSON imbriquées du NVD,  
puis jointure avec la vérité terrain CISA KEV pour labelliser `is_exploited`.

In [ ]:
# ==============================================================================
# CELLULE 5 — Parsing du JSON NVD et construction du DataFrame
# ==============================================================================

def parse_nvd_to_dataframe(nvd_data: List[Dict[str, Any]],
                            cisa_cves: set) -> pd.DataFrame:
    """
    Transforme la liste brute d'objets CVE (NVD JSON) en DataFrame structuré.

    - Extrait les métriques CVSS v3.1 (ou v3.0 si v3.1 absent).
    - Consolide les CWE multiples d'une même CVE en chaîne séparée par des virgules.
    - Labellise `is_exploited` via le catalogue CISA KEV.

    Parameters
    ----------
    nvd_data : list[dict]
        Objets CVE bruts retournés par fetch_nvd_api().
    cisa_cves : set[str]
        Ensemble des CVE-IDs activement exploités (source CISA KEV).

    Returns
    -------
    pd.DataFrame
        Dataset nettoyé avec CVSS v3.1 + label binaire.
    """
    records = []

    for cve in nvd_data:
        cve_id  = cve.get('id')
        metrics = cve.get('metrics', {})

        # ── Extraction des CWE associés ──────────────────────────────────────
        # Une CVE peut appartenir à plusieurs catégories CWE simultanément.
        cwe_ids = sorted({
            desc.get('value')
            for weakness in cve.get('weaknesses', [])
            for desc in weakness.get('description', [])
            if desc.get('lang') == 'en' and str(desc.get('value', '')).startswith('CWE-')
        })

        # ── Sélection des métriques CVSS (priorité v3.1 > v3.0) ─────────────
        cvss_configs = metrics.get('cvssMetricV31', []) or metrics.get('cvssMetricV30', [])
        if not cvss_configs:
            continue  # CVE sans CVSS v3 : exclue du corpus

        cvss = cvss_configs[0].get('cvssData', {})

        records.append({
            'cve_id'                : cve_id,
            'cwe_id'                : ','.join(cwe_ids),
            'published_date'        : cve.get('published'),
            'base_score'            : cvss.get('baseScore'),
            'attack_vector'         : cvss.get('attackVector'),
            'attack_complexity'     : cvss.get('attackComplexity'),
            'privileges_required'   : cvss.get('privilegesRequired'),
            'user_interaction'      : cvss.get('userInteraction'),
            'scope'                 : cvss.get('scope'),
            'confidentiality_impact': cvss.get('confidentialityImpact'),
            'integrity_impact'      : cvss.get('integrityImpact'),
            'availability_impact'   : cvss.get('availabilityImpact'),
            'is_exploited'          : int(cve_id in cisa_cves),
        })

    df = pd.DataFrame(records)

    # ── Déduplication (une même CVE peut apparaître dans plusieurs CWE) ──────
    df = df.drop_duplicates(subset='cve_id').reset_index(drop=True)

    return df


df = parse_nvd_to_dataframe(raw_nvd_data, cisa_exploited_cves)

print(f'[+] Dataset après parsing       : {df.shape[0]:,} CVE × {df.shape[1]} colonnes')
print(f'[+] CVE exploitées (Y=1)        : {df["is_exploited"].sum():,} '
      f'({df["is_exploited"].mean()*100:.2f} %)')
print(f'[+] Répartition par CWE primaire :')
primary_cwe = df['cwe_id'].str.split(',').str[0]
print(primary_cwe.value_counts().to_string())

## Section 4 — Feature Engineering

Création de la feature temporelle `cve_age_days` : nombre de jours écoulés  
depuis la date de publication de la CVE jusqu'à la date de référence.  
Cette variable capture la fenêtre d'exposition temporelle de la vulnérabilité.

In [ ]:
# ==============================================================================
# CELLULE 6 — Feature temporelle : âge de la CVE
# ==============================================================================

# ── Nettoyage des données manquantes CVSS ────────────────────────────────────
# Les lignes sans métriques critiques sont exclues pour garantir l'intégrité
# de la matrice d'apprentissage (pas d'imputation sur des features nominales CVSS).
df_clean = df.dropna(subset=CVSS_FEATURES).copy()
n_dropped = len(df) - len(df_clean)
if n_dropped > 0:
    logging.info(f'  {n_dropped} CVE supprimées (CVSS v3 incomplet)')

# ── Calcul de l'âge de la CVE ────────────────────────────────────────────────
df_clean['published_date'] = pd.to_datetime(df_clean['published_date'], errors='coerce')
df_clean['cve_age_days'] = (
    (REFERENCE_DATE - df_clean['published_date']).dt.days
)

# Imputation par la médiane pour les dates manquantes (< 0.1 % des cas)
median_age = df_clean['cve_age_days'].median()
df_clean['cve_age_days'] = df_clean['cve_age_days'].fillna(median_age).astype(int)

print(f'[+] Feature temporelle créée : cve_age_days')
print(f'    Date de référence : {REFERENCE_DATE.date()}')
print(f'    Médiane           : {median_age:.0f} j')
print(f'    Min / Max         : {df_clean["cve_age_days"].min()} j / {df_clean["cve_age_days"].max()} j')
print(f'    Corrélation avec is_exploited : {df_clean["cve_age_days"].corr(df_clean["is_exploited"]):.4f}')

## Section 5 — Validation et Sauvegarde

Contrôle de qualité final avant l'écriture sur disque.  
Le CSV n'est écrit qu'**une seule fois**, ici, après tous les traitements.

In [ ]:
# ==============================================================================
# CELLULE 7 — Sélection des colonnes finales, validation et sauvegarde
# ==============================================================================

# ── Sélection des colonnes exportées ─────────────────────────────────────────
FINAL_COLUMNS = [
    'cve_id', 'cwe_id', 'published_date', 'base_score',
    'attack_vector', 'attack_complexity', 'privileges_required',
    'user_interaction', 'scope',
    'confidentiality_impact', 'integrity_impact', 'availability_impact',
    'is_exploited', 'cve_age_days'
]

df_final = df_clean[FINAL_COLUMNS].copy()

# ── Validation avant sauvegarde ──────────────────────────────────────────────
assert df_final['cve_id'].is_unique, 'Doublons détectés dans cve_id — pipeline corrompu'
assert df_final['is_exploited'].isin([0, 1]).all(), 'Valeurs invalides dans is_exploited'
missing_cvss = df_final[CVSS_FEATURES].isna().sum().sum()
assert missing_cvss == 0, f'{missing_cvss} valeurs CVSS manquantes — nettoyage incomplet'

# ── Sauvegarde (écriture unique — ne jamais écrire dans les cellules précédentes) ──
df_final.to_csv(OUTPUT_CSV, index=False)

print(f'[✓] Dataset sauvegardé : {OUTPUT_CSV}')
print(f'    Dimensions  : {df_final.shape[0]:,} lignes × {df_final.shape[1]} colonnes')
print(f'    Colonnes    : {list(df_final.columns)}')
print()
print('─' * 55)
print('STATISTIQUES DESCRIPTIVES')
print('─' * 55)
print(f'  N total            : {len(df_final):,}')
print(f'  Y=1 (exploitées)   : {df_final["is_exploited"].sum():,} '
      f'({df_final["is_exploited"].mean()*100:.2f} %)')
print(f'  Y=0 (non exploitées): {(df_final["is_exploited"] == 0).sum():,}')
print()
print('  Distribution attack_vector :')
print(df_final['attack_vector'].value_counts().to_string(header=False))
print()
print('  Top 5 CWE primaires :')
print(df_final['cwe_id'].str.split(',').str[0].value_counts().head(5).to_string(header=False))